**1.prepare DATA**

In [ ]:
!pip install ultralytics -q

from google.colab import drive
import os
import zipfile
import yaml
import torch
import glob
from ultralytics import YOLO
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# =====================================================================
# การจัดการไฟล์ (Mount Drive แบบบังคับรีเฟรช + ตรวจสอบ Path ทีละชั้น + แตกไฟล์ ZIP)
# =====================================================================
import os
import shutil
import zipfile
from google.colab import drive

# 1. เช็กที่ MyDrive โดยตรง (ไม่ใช่แค่ /content/drive)
if not os.path.exists('/content/drive/MyDrive'):
    drive.mount('/content/drive', force_remount=True)

FOLDER_PATH = "/content/drive/MyDrive/rmutl-pj"
ZIP_NAME = "DATA-Ultralytics_20261124_124921_28186ebb.zip"
ZIP_PATH = os.path.join(FOLDER_PATH, ZIP_NAME)
LOCAL_BASE = "/content/dataset"

# 2. หากยังไม่เจอ ให้บังคับ remount อีกรอบ (แก้ปัญหาเพิ่งอัปโหลดไฟล์แล้ว Drive ยังไม่อัปเดตแคช)
if not os.path.exists(ZIP_PATH):
    print("🔄 กำลังรีเฟรชการเชื่อมต่อ Google Drive...")
    drive.mount('/content/drive', force_remount=True)

# 3. ตรวจสอบทีละชั้นและแตกไฟล์
if os.path.exists(ZIP_PATH):
    if os.path.exists(LOCAL_BASE):
        print("⚠️ พบโฟลเดอร์ Dataset เก่า → กำลังล้างข้อมูลเก่า")
        shutil.rmtree(LOCAL_BASE)

    print(f"📦 กำลังแตกไฟล์จาก: {ZIP_PATH}")
    os.makedirs(LOCAL_BASE, exist_ok=True)
    with zipfile.ZipFile(ZIP_PATH, 'r') as zip_ref:
        zip_ref.extractall(LOCAL_BASE)
    print("✅ แตกไฟล์ลงดิสก์ชั่วคราว (/content/dataset) เรียบร้อย!")

elif os.path.exists(FOLDER_PATH):
    # เจอโฟลเดอร์ Pre-Project แต่ชื่อไฟล์ไม่ตรง -> แสดงชื่อไฟล์จริงทั้งหมดที่มีในโฟลเดอร์
    print(f"⚠️ เจอโฟลเดอร์ {FOLDER_PATH} แต่ไม่พบไฟล์ชื่อ '{ZIP_NAME}'")
    print("📂 รายชื่อไฟล์ทั้งหมดที่มีอยู่จริงในโฟลเดอร์นี้:")
    for f in sorted(os.listdir(FOLDER_PATH)):
        print(f"   - {repr(f)}")
else:
    print(f"❌ ไม่พบโฟลเดอร์: {FOLDER_PATH}")
    if os.path.exists('/content/drive/MyDrive'):
        print("📂 โฟลเดอร์ที่ใกล้เคียงใน MyDrive:")
        for d in sorted(os.listdir('/content/drive/MyDrive')):
            if "pre" in d.lower() or "project" in d.lower():
                print(f"   - {repr(d)}")

📦 กำลังแตกไฟล์จาก: /content/drive/MyDrive/rmutl-pj/DATA-Ultralytics_20261124_124921_28186ebb.zip
✅ แตกไฟล์ลงดิสก์ชั่วคราว (/content/dataset) เรียบร้อย!


### 2. ตรวจสอบจำนวนไฟล์ใน Dataset เพื่อยืนยันความถูกต้อง 100%

In [ ]:
import glob
import os
import yaml

# กำหนดค่าเริ่มต้น
LOCAL_BASE = "/content/dataset"
yaml_path = "/content/dataset/data.yaml"

# โหลด data_yaml ในเซลล์นี้เลยเพื่อป้องกันข้อผิดพลาด NameError
if os.path.exists(yaml_path):
    with open(yaml_path, 'r', encoding='utf-8') as f:
        data_yaml = yaml.safe_load(f)

    # กำหนด path ของภาพฝึกสอนและทดสอบตามค่าใน data.yaml
    train_images_path = os.path.join(LOCAL_BASE, data_yaml['train'])
    val_images_path = os.path.join(LOCAL_BASE, data_yaml['val'])

    # ค้นหาไฟล์รูปภาพทั้งหมด (รองรับหลายนามสกุลไฟล์)
    extensions = ('*.jpg', '*.jpeg', '*.png', '*.bmp', '*.JPG', '*.JPEG', '*.PNG')

    train_images = []
    for ext in extensions:
        train_images.extend(glob.glob(os.path.join(train_images_path, ext)))

    val_images = []
    for ext in extensions:
        val_images.extend(glob.glob(os.path.join(val_images_path, ext)))

    print(f"📊 จำนวนไฟล์ภาพที่ตรวจพบจริงบนดิสก์:\n")

    # แสดงจำนวนและรายชื่อไฟล์ในชุด Train
    print(f"🔹 โฟลเดอร์ Train ({data_yaml['train']}): พบ {len(train_images)} ภาพ")
    if train_images:
        print("   รายชื่อไฟล์:")
        for img_path in sorted(train_images):
            print(f"   - {os.path.basename(img_path)}")
    else:
        print("   (ไม่พบไฟล์ภาพ)")

    print("\n" + "=" * 50 + "\n")

    # แสดงจำนวนและรายชื่อไฟล์ในชุด Val
    print(f"🔹 โฟลเดอร์ Val ({data_yaml['val']}): พบ {len(val_images)} ภาพ")
    if val_images:
        print("   รายชื่อไฟล์:")
        for img_path in sorted(val_images):
            print(f"   - {os.path.basename(img_path)}")
    else:
        print("   (ไม่พบไฟล์ภาพ)")

    print("\n" + "=" * 50)
    print(f"📦 รวมทั้งหมด: {len(train_images) + len(val_images)} ภาพ")
    print("\n💡 หากตัวเลขข้างต้นตรงกับข้อมูลชุดข้อมูลของคุณ แสดงว่าโมเดลจะดึงข้อมูลภาพไปใช้งานได้ครบถ้วนสมบูรณ์ครับ!")
else:
    print(f"❌ ไม่พบไฟล์ data.yaml ที่ {yaml_path} กรุณารันขั้นตอนแตกไฟล์และอัปเดตข้อมูลก่อนหน้านี้ก่อนครับ")

📊 จำนวนไฟล์ภาพที่ตรวจพบจริงบนดิสก์:

🔹 โฟลเดอร์ Train (images/train): พบ 510 ภาพ
   รายชื่อไฟล์:
   - 2026-11-24__001_camera_20261124_075856_033926_frame_000012_00000055.000s.jpg
   - 2026-11-24__001_camera_20261124_075856_033926_frame_000013_00000060.000s.jpg
   - 2026-11-24__001_camera_20261124_075856_033926_frame_000014_00000065.000s.jpg
   - 2026-11-24__001_camera_20261124_075856_033926_frame_000015_00000070.000s.jpg
   - 2026-11-24__002_camera_20261124_082210_854107_frame_000001_00000000.000s.jpg
   - 2026-11-24__002_camera_20261124_082210_854107_frame_000002_00000005.000s.jpg
   - 2026-11-24__002_camera_20261124_082210_854107_frame_000003_00000010.000s.jpg
   - 2026-11-24__002_camera_20261124_082210_854107_frame_000004_00000015.000s.jpg
   - 2026-11-24__002_camera_20261124_082210_854107_frame_000005_00000020.000s.jpg
   - 2026-11-24__002_camera_20261124_082210_854107_frame_000006_00000025.000s.jpg
   - 2026-11-24__002_camera_20261124_082210_854107_frame_000007_00000030.000s.jpg
 

### 3. เริ่มต้นการเทรนโมเดล YOLO (Training)

In [ ]:
import os
import yaml

yaml_path = "/content/dataset/data.yaml"
base_dir = "/content/dataset"

# 1. อ่านไฟล์ data.yaml เดิม
with open(yaml_path, 'r', encoding='utf-8') as f:
    data_yaml = yaml.safe_load(f)

# 2. ปรับ path ให้ชี้ไปที่โฟลเดอร์บน Colab และตรวจหาโฟลเดอร์ train/val ที่มีอยู่จริง
data_yaml['path'] = base_dir

train_candidates = ['images/train', 'train', 'train/images', 'images/train/images']
for candidate in train_candidates:
    if os.path.exists(os.path.join(base_dir, candidate)):
        data_yaml['train'] = candidate
        break

val_candidates = ['images/val', 'val', 'valid', 'valid/images', 'images/valid']
for candidate in val_candidates:
    if os.path.exists(os.path.join(base_dir, candidate)):
        data_yaml['val'] = candidate
        break

# 3. บันทึกไฟล์กลับลงไป
with open(yaml_path, 'w', encoding='utf-8') as f:
    yaml.safe_dump(data_yaml, f, sort_keys=False)

print("✅ อัปเดต data.yaml เรียบร้อยแล้ว!")
print(f"path  = {data_yaml.get('path')}")
print(f"train = {data_yaml.get('train')}")
print(f"val   = {data_yaml.get('val')}")

✅ อัปเดต data.yaml เรียบร้อยแล้ว!
path  = /content/dataset
train = images/train
val   = images/val


In [ ]:
from ultralytics import YOLO
import torch

# โหลดโมเดลเดิมมาเทรนต่อกับ Dataset ชุดที่แก้และคัดภาพซ้ำแล้ว
model = YOLO("/content/3best.pt")

device = 0 if torch.cuda.is_available() else 'cpu'
print(f"🔥 Training with MuSGD on: {device}")

results = model.train(
    # --- Basic Settings ---
    data="/content/dataset/data.yaml",
    epochs=150,                          # MuSGD ลู่เข้าไวกว่า SGD ปกติ ใช้ 150 รอบเพียงพอ
    imgsz=640,
    batch=16,                            # แนะนำล็อกค่า batch (เช่น 16 หรือ 32) เพื่อความนิ่งของการคำนวณ

    # --- Save Options ---
    project="/content/drive/MyDrive/model-preple-detect/runs",
    name="T4_yolo_musgd",                # แยกชื่อโฟลเดอร์ไว้เทียบผลกับ T1
    save_period=10,

    # --- Optimization Tricks for MuSGD Fine-tuning ---
    optimizer='MuSGD',                   # ใช้ MuSGD (Muon + SGD)
    lr0=0.0005,                          # ลดจาก 0.01 ลง 20 เท่า เพื่อถนอมน้ำหนักเดิมของ 1best.pt
    lrf=0.01,
    warmup_epochs=1.0,                   # ลดช่วง Warmup เหลือ 1 รอบ
    patience=40,
    device=device
)

🔥 Training with MuSGD on: 0
Ultralytics 8.4.218 🚀 Python-3.13.16 torch-2.14.1+cu130 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, channels_last=None, classes=None, close_mosaic=10, cls=0.5, cls_pw=0.0, cls_remap=True, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/content/dataset/data.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dgrad=0.5, dis=6.0, distill_model=None, dlam=1.0, dlog=1.0, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=150, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, iou=None, kobj=1.0, line_width=None, lr0=0.0005, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/content/3best.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=T4_yolo_musgd-3, nbs=64